# 📚 Class 5: Neo Reads the Policy Binder
### Giving AI our knowledge (RAG) · Day 3 · Morning · 2 hours

**Same story as Day 2.** Neo, your AI chatbot, covers your inbox while you're on leave. Today a vendor asks a question that is **only answered in your company's policy binder**. Neo has never seen that binder.

**Today's question:** *How do we make Neo answer from our documents, and show where the answer came from?*

| Part | Topic | Cells |
|---|---|---|
| 1 | Neo doesn't know our binder | S, 1–3 |
| 2 | The open-book exam: the 4 steps of RAG | 4–9 |
| 3 | Trust but verify: refusing, testing, outdated documents | 10–13 |
| 4 | Your turn | 14–15 |

> 💡 **Key terms come first.** Each part starts with a key-terms slide. Then you run the cells.
> 🔒 All documents are **mock** (fictional company *Nova Industries*). Everything runs on this laptop with Ollama.

## ⚙️ ▶ CELL S: Setup
Checks that Ollama is running and the models (chat + embedding) are downloaded. Run once.

In [ ]:
# ▶ CELL S · SETUP: check Ollama (chat model + embedding model). Run once.
import sys, os, re, json, glob
sys.path.insert(0, "../../shared")            # where bootcamp_llm.py lives
from bootcamp_llm import *

DOCS_DIR = "../data/docs"                     # the mock company documents (Nova Industries)
check_setup(need_embed=True)
print("🤖 Neo says:", ask_ai("Reply with exactly these words: Connection OK", temperature=0, max_tokens=10))

---
# 1️⃣ Neo doesn't know our binder
**In real life:** a new colleague on day one is asked *"who approves a PO above ₹10 lakh?"*. They haven't read the policy yet. A careful one says "I'll check". A confident one guesses.

### ▶ CELL 1: Ask Neo, with no documents

In [ ]:
# ▶ CELL 1 · No documents: Neo answers from general knowledge
QUESTION = "In our company, who approves a purchase order above ₹10 lakh?"   # ← change me
show(ask_ai(QUESTION, temperature=0))
print(f"({last_usage['seconds']} s)")

> 🔎 **What to look for:** a generic answer ("usually a senior manager or the CFO…"). It sounds fine, but **it is not our policy**. Our policy says *Director* (Section 3.3). Better prompts can't fix this: Neo never saw the document.

### ▶ CELL 2: Paste the whole policy into the prompt
This is the simplest fix, and it works for **one short document**.

In [ ]:
# ▶ CELL 2 · Paste the whole policy into the prompt ("context stuffing")
policy = open(f"{DOCS_DIR}/Procurement_Policy.txt", encoding="utf-8").read()
prompt = f"""Answer ONLY from the policy below. If the answer is not in it, say "Not in the documents."

POLICY:
<<<
{policy}
>>>

Question: {QUESTION}"""
show(ask_ai(prompt, temperature=0))

total_words = sum(len(open(f, encoding="utf-8").read().split()) for f in glob.glob(f"{DOCS_DIR}/*.txt"))
print(f"\nThis prompt: {len(prompt.split())} words.   All 4 mock documents together: {total_words} words.")
print("A real company has hundreds of documents. They will not all fit in the context window, and every question would pay for all of them.")

> ✅ With the policy in front of it, Neo answers correctly. **But we can't paste 500 documents every time.** We need to paste only the *right page*. That is RAG.

### ▶ CELL 3: Where does "memory" come from?
The model remembers nothing between messages. The **app** re-sends the conversation each time.

In [ ]:
# ▶ CELL 3 · Chat "memory": the APP re-sends the history, the model remembers nothing
q1 = {"role": "user", "content": "My PO number is 4500012345. Please just say OK."}
a1 = chat([q1], temperature=0)
q2 = {"role": "user", "content": "What was my PO number?"}
print("A) Only the new message sent :", chat([q2], temperature=0))
print("B) Whole history re-sent     :", chat([q1, {"role": "assistant", "content": a1}, q2], temperature=0))

> 🧠 In **A** the model has no idea. In **B** it knows, because we sent the earlier messages again. *Memory is something the app does, not the model.*

---
# 2️⃣ The open-book exam: RAG in 4 steps
**In real life:** closed-book = answer from memory (and guess when unsure). Open-book = **find the right page first, then answer from it**.

```
 1. CUT      →   2. STORE      →   3. FIND       →   4. ANSWER
 split docs      save by           get the best      give pieces +
 into pieces     MEANING           pieces for a      question to Neo
 (chunks)        (embeddings)      question          → cited answer
```

### ▶ CELL 4: Step 1, CUT the documents into chunks
We cut **one chunk per numbered clause** and label each with its source, so every answer can say *where it came from*.

In [ ]:
# ▶ CELL 4 · STEP 1 · CUT: split the documents into chunks
def cut(path):
    """Split one document into chunks: one chunk per numbered clause, labelled with its source and section."""
    name = os.path.basename(path).replace(".txt", "")
    heading, chunks = "", []
    for block in open(path, encoding="utf-8").read().split("\n\n"):
        for line in block.strip().splitlines():
            m = re.match(r"^(\d+\.\d+)\s+(.*)", line)
            if m:                                    # a numbered clause, e.g. "3.3 POs above ..."
                chunks.append({"id": f"{name} §{m.group(1)}", "text": f"[{name} · {heading}] {line}"})
            elif re.match(r"^\d+\.\s", line):        # a section heading, e.g. "3. PO APPROVAL LIMITS"
                heading = line
    return chunks

PACKS = {"A": ["Procurement_Policy.txt", "SAP_P2P_Process_Guide.txt"],   # A = Procurement
         "B": ["Travel_Policy.txt"],                                      # B = Travel
         "C": ["IDoc_Error_Guide.txt"]}                                   # C = IDoc errors
DOC_PACK = "A"                                                            # ← change me (A, B or C)

chunks = [c for name in PACKS[DOC_PACK] for c in cut(f"{DOCS_DIR}/{name}")]
print(f"Pack {DOC_PACK}: {len(chunks)} chunks. The first three:\n")
for c in chunks[:3]:
    print(" •", c["id"], "→", c["text"][:95], "…")
print("\nThe chunk that holds the Director rule:")
print(" •", next(c["text"] for c in chunks if c["id"].endswith("§3.3")))

> ✂️ **Chunk** = a small piece of a document. We kept each clause whole so the meaning isn't cut in half.

### ▶ CELL 5: Why not just search for words?
Normal (keyword) search counts matching words. Watch what happens when the question uses **different words** from the policy.

In [ ]:
# ▶ CELL 5 · Keyword search: finds matching WORDS, not matching MEANING
STOP = set("a an the of to is are who what which and or in on for do does it be by at as from that this with must".split())
def words_of(text): return set(re.findall(r"[a-z0-9]+", text.lower())) - STOP

def keyword_search(question, top=3):
    q = words_of(question)
    scored = sorted(((len(q & words_of(c["text"])), c) for c in chunks), key=lambda s: -s[0])
    return [(score, c) for score, c in scored[:top] if score > 0]

for question in ["Who approves a PO above 10 lakh?", "Who signs off on really big orders?"]:
    print("❓", question)
    for score, c in keyword_search(question):
        print(f"   {score} matching words → {c['id']}")
    print()

> 😕 The reworded question *"Who signs off on really big orders?"* shares almost no words with Section 3.3 (*"POs above ₹10,00,000 require Director approval"*). Keyword search picks the wrong clause. **We need search by meaning.**

### ▶ CELL 6: Step 2, STORE chunks as "meaning fingerprints"
An **embedding** is a list of numbers that captures what a text *means*. Similar meaning → similar numbers.

In [ ]:
# ▶ CELL 6 · STEP 2 · STORE: turn every chunk into an embedding (a "meaning fingerprint")
fingerprints = embed([c["text"] for c in chunks])
store = list(zip(chunks, fingerprints))
print(f"Stored {len(store)} chunks. Each fingerprint is a list of {len(fingerprints[0])} numbers.")
print("First 6 numbers of one fingerprint:", [round(x, 3) for x in fingerprints[0][:6]])

# Similar meaning → similar numbers (1.0 = identical meaning, near 0 = unrelated)
a, b, c_ = embed(["Who approves a big purchase order?",
                  "Director approval is needed above ₹10 lakh",
                  "Payment runs happen on Tuesday and Friday"])
print("\nbig PO question  ↔  Director approval rule :", round(cosine(a, b), 2))
print("big PO question  ↔  Tuesday payment run    :", round(cosine(a, c_), 2))

> 🧭 The first pair scores higher than the second. That is all "search by meaning" is: **compare fingerprints**. (Embeddings are *not* encryption; they are only a way to search.)

### ▶ CELL 7: Step 3, FIND the best chunks

In [ ]:
# ▶ CELL 7 · STEP 3 · FIND: compare the question's fingerprint with every chunk
def find(question, top=3, the_store=None):
    the_store = the_store or store
    q = embed(question)
    ranked = sorted(the_store, key=lambda s: cosine(q, s[1]), reverse=True)[:top]
    return [(round(cosine(q, v), 2), c) for c, v in ranked]

for question in ["Who approves a PO above ₹10 lakh?", "Who signs off on really big orders?"]:
    print("❓", question)
    for score, c in find(question):
        print(f"   {score}  {c['id']:<32} {c['text'][-60:]}")
    print()

> 🎯 Now the reworded question should find **Section 3.3** by meaning, with no shared keywords. (A small model may rank it 2nd or 3rd. That is why we pass the **top 3** chunks, not just one.)

### ▶ CELL 8: Step 4, ANSWER from the pieces, with a citation

In [ ]:
# ▶ CELL 8 · STEP 4 · ANSWER: give the best chunks + the question to Neo
RAG_RULES = """You are Neo, answering questions for Nova Industries staff.
Answer ONLY from the CONTEXT. If the answer is not in the CONTEXT, reply exactly: Not in the documents.
Keep it to one or two sentences and end with the source in brackets, for example [Procurement_Policy §3.3]."""

def rag(question, top=3, the_store=None, rules=RAG_RULES):
    hits = find(question, top, the_store)
    context = "\n".join(f"[{c['id']}] {c['text']}" for _, c in hits)
    prompt = f"CONTEXT:\n<<<\n{context}\n>>>\n\nQuestion: {question}"
    return ask_ai(prompt, system=rules, temperature=0), hits

answer, hits = rag("Who approves a PO above ₹10 lakh?")
print("🤖 Neo:", answer)
print("📎 Sources it was given:", [c["id"] for _, c in hits])

> ✅ **Retrieval + Augmented + Generation.** Neo answered from our document and named the source. *Click-to-verify is the whole point.*

### ▶ CELL 9: The reworded question, end to end

In [ ]:
# ▶ CELL 9 · The reworded question, end to end
for question in ["Who signs off on really big orders?",
                 "What must happen before a vendor's bank details can be changed?"]:
    answer, hits = rag(question)
    print("❓", question)
    print("🤖", answer)
    print("📎", [c["id"] for _, c in hits], "\n")

> 💡 The second question connects to **Email D** from Day 2 (the bank-detail scam). Our policy requires *written confirmation AND a call-back* (Section 5.2). RAG lets Neo apply **our** rule.

---
# 3️⃣ Trust but verify
### ▶ CELL 10: What if the answer is NOT in the documents?
A grounded assistant should say **"Not in the documents"**, not guess. Compare a strict rule with a weak one.

In [ ]:
# ▶ CELL 10 · The "I don't know" test: not in the documents
NOT_THERE = "What is the daily allowance for a trip to Singapore?"          # ← change me
WEAK_RULES = "You are Neo, a helpful assistant for Nova Industries. Use the CONTEXT if it helps."

strict, hits = rag(NOT_THERE)
weak, _ = rag(NOT_THERE, rules=WEAK_RULES)
print("❓", NOT_THERE, "(pack A has no travel policy)\n")
print("🔒 With a strict rule :", strict)
print("🔓 With a weak rule   :", weak)

> 🛡️ The strict rule should give **"Not in the documents."** The weak rule may produce a confident number from nowhere. *The rule in the note is part of the system, not decoration.*

### ▶ CELL 11: Test five questions and score them

In [ ]:
# ▶ CELL 11 · Test plan: answerable · reworded · not in the documents. The CODE checks the retrieval.
TESTS = [  # (type, question, the clause that must be retrieved, or None if the answer is not in the documents)
    ("direct",   "Who approves a PO above ₹10 lakh?",                             "Procurement_Policy §3.3"),
    ("direct",   "What must happen before a vendor's bank details can be changed?", "Procurement_Policy §5.2"),
    ("direct",   "Which transaction posts a goods receipt?",                      "SAP_P2P_Process_Guide §1.3"),
    ("reworded", "Who signs off on really big orders?",                           "Procurement_Policy §3.3"),
    ("missing",  "What is the daily allowance for a trip to Singapore?",          None),
]

def score_tests(tests, the_store=None):
    points = 0
    for kind, question, expected in tests:
        answer, hits = rag(question, the_store=the_store)
        ids = [c["id"] for _, c in hits]
        ok = (expected in ids) if expected else ("not in the documents" in answer.lower())
        points += ok
        print(f"{'✅' if ok else '❌'} {kind:<8} {question}")
        print(f"     retrieved: {ids}")
        print(f"     answer   : {answer[:150]}\n")
    print(f"Score: {points} / {len(tests)}")

score_tests(TESTS)

> 📋 Always test **all three kinds**: a direct question, a reworded one, and one that is *not* in the documents. A ❌ is a good lesson: was the chunk not retrieved, or did Neo ignore it?

### ▶ CELL 12: The outdated-document trap

In [ ]:
# ▶ CELL 12 · Outdated document: RAG quotes whatever you give it
old_chunks = cut("../data/old/Procurement_Policy_2024_OLD.txt")          # the 2024 policy: limits were lower
old_store = store + list(zip(old_chunks, embed([c["text"] for c in old_chunks])))

QUESTION_7L = "Who approves a PO of ₹7,00,000?"
print("Current policy says: above ₹1,00,000 and up to ₹10,00,000 → Department Head (Section 3.2).")
print("2024 policy says   : above ₹5,00,000 → Director.\n")

answer, hits = rag(QUESTION_7L, the_store=old_store)
print("📚 With BOTH versions stored → retrieved:", [c["id"] for _, c in hits])
print("🤖", answer, "\n")
answer, hits = rag(QUESTION_7L)
print("📗 Current policy only      → retrieved:", [c["id"] for _, c in hits])
print("🤖", answer)

> ⚠️ Two versions = a coin toss, or a confidently wrong answer. **RAG is only as good as the documents.** The fix is not a better prompt: it is keeping the library current (remove old versions or tag them).

### ▶ CELL 13: Chunk size matters

In [ ]:
# ▶ CELL 13 · Chunk size: one big chunk per document vs one chunk per clause
big_docs = [{"id": n.replace(".txt", ""), "text": open(f"{DOCS_DIR}/{n}", encoding="utf-8").read()} for n in PACKS["A"]]
big_store = list(zip(big_docs, embed([d["text"] for d in big_docs])))

q = "Who approves a PO above ₹10 lakh?"
ans_big, hits_big = rag(q, top=1, the_store=big_store);   tokens_big = last_usage["input_tokens"]
ans_small, hits_small = rag(q, top=3);                    tokens_small = last_usage["input_tokens"]

print(f"BIG chunks   → cites: {[c['id'] for _, c in hits_big]}   prompt size: {tokens_big} tokens")
print(f"SMALL chunks → cites: {[c['id'] for _, c in hits_small]}   prompt size: {tokens_small} tokens")
print("\nBig chunks can only cite a whole document. Small chunks cite the exact clause and keep the prompt short.")

---
# 4️⃣ Your turn (40 min)
Change only the lines marked **← change me**, then press **Shift + Enter**.

### ▶ CELL 14: Try another document pack
Pack **B** = Travel policy · Pack **C** = IDoc error handbook.

In [ ]:
# ▶ CELL 14 · Your turn: another document pack
MY_PACK = "C"                                              # ← change me ("B" = Travel, "C" = IDoc errors)
my_chunks = [c for name in PACKS[MY_PACK] for c in cut(f"{DOCS_DIR}/{name}")]
my_store = list(zip(my_chunks, embed([c["text"] for c in my_chunks])))
print(f"Pack {MY_PACK}: {len(my_chunks)} chunks stored.\n")

MY_QUESTION = "What does IDoc status 51 mean?"            # ← change me (pack B: "What is the daily allowance in Singapore?")
answer, hits = rag(MY_QUESTION, the_store=my_store)
print("🤖", answer)
print("📎", [c["id"] for _, c in hits])

### ▶ CELL 15: Write your own five test questions and score them
Use your pack from Cell 14. For each question write the clause that **must** come back (look in `data/docs/`). Use `None` for a question that is not in the documents.

In [ ]:
# ▶ CELL 15 · Your turn: your own 5 test questions (3 direct · 1 reworded · 1 not in the documents)
MY_TESTS = [   # ← change me: replace these examples (they are for pack C)
    ("direct",   "What does IDoc status 56 mean?",                                  "IDoc_Error_Guide §1.2"),
    ("direct",   "Which transaction is used to reprocess failed IDocs?",            "IDoc_Error_Guide §2.2"),
    ("direct",   "When should the Basis team be involved?",                         "IDoc_Error_Guide §2.3"),
    ("reworded", "A pile of IDocs is stuck waiting to be processed. Where do I look?", "IDoc_Error_Guide §1.4"),
    ("missing",  "What is the approval limit for a purchase order?",                None),
]
score_tests(MY_TESTS, the_store=my_store)

---
## ✅ Remember three things
1. **RAG = open-book exam:** find the right page first, then answer from it.
2. **Citations** make answers checkable. **"Not in the documents"** is a correct answer.
3. **RAG is only as good as your documents:** current, approved, non-sensitive.

> 🌉 **Bridge to Class 6:** Neo can now *answer* from our documents. But when a manager asks *"Why is PO 4500012345 delayed and who should I contact?"* the answer isn't in a document. Neo has to **check SAP, look up the vendor and apply the policy**. Can Neo **do things**?

---
## 🛠️ Trainer troubleshooting (Ollama)
| Message | Cause | Fix |
|---|---|---|
| `Cannot reach Ollama at http://localhost:11434` | Ollama isn't running | Windows: Start menu → **Ollama**. Mac: open the Ollama app. Or in a terminal: `ollama serve` |
| `Model '…' is not downloaded` | The model in `bootcamp_config.json` isn't pulled | `ollama pull <name>` (see `ollama list`) |
| `ModuleNotFoundError: bootcamp_llm` | The notebook is not running from its own folder | Open the **Day_3** folder as the workspace, then pick the **`.venv`** kernel |
| Very slow answers | The model is too big for this laptop | Choose a smaller `model` in `bootcamp_config.json`, then **Restart kernel** |
| `The model did not return valid JSON` (Class 6) | Small models sometimes break the format | Run the cell again. This is a real lesson: agents need validation |
| `embed` fails or all scores look the same (Class 5) | The embedding model is missing | `ollama pull nomic-embed-text`, or set `embed_model` to your chat model in the config |